# 🚀 VIDEO-FREE · 7. Director en Kaggle (2 GPU gratis)

Kaggle da gratis **2 GPU T4** a la vez (cuota semanal de unas 30 horas). Este cuaderno pone a trabajar las dos en tu video. Si además corre el cuaderno 5 en Colab, son **3 GPU** repartiéndose las tomas a través de tu repo de GitHub.

**Preparar Kaggle (una sola vez):**
1. Crea una cuenta gratis en kaggle.com y **verifica tu teléfono** (Settings → Phone verification). Sin eso no hay GPU ni internet.
2. Descarga `colab/7_kaggle_director.ipynb` de tu repo en GitHub y en Kaggle haz *Create → New Notebook → File → Import Notebook* y súbelo.
3. En el panel derecho, *Session options*: **Accelerator → GPU T4 x2** e **Internet → On**.
4. *Add-ons → Secrets → Add secret*: nombre `GITHUB_TOKEN`, el mismo token de Colab. Marca la casilla para que este cuaderno lo use.

**Usarlo:**
- **Con Colab a la vez (3 GPU):** aquí deja `TRABAJADORES = 3` y `PRIMER_TRABAJADOR = 2`. En Colab pon `TRABAJADORES = 3` y `TRABAJADOR = 1`. Usa las mismas opciones en los dos lados. Colab monta el video al final.
- **Solo Kaggle (2 GPU):** pon `TRABAJADORES = 2`, `PRIMER_TRABAJADOR = 1` y `MONTAR_AQUI = True`.

Luego *Run All*. Kaggle no tiene Google Drive: todo se comparte por tu repo.


In [ ]:
# 1 · Opciones (iguales a las del cuaderno 5)
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"
RAMA = "colab-ia"
PROYECTO = "carretera-sin-ti.json"
CALIDAD = "borrador"          # "borrador" (480p) o "final"
FORMATO = "horizontal"        # "horizontal" o "vertical"
MOTOR = "rapido"              # "rapido" (FastWan, 3 pasos) o "calidad" (Wan, 30 pasos)
MODO_PERSONAJE = "recorte"    # "recorte" o "entrenado"
LABIOS = True
RITMO = "auto"
SEMILLA = 11
TRABAJADORES = 3              # 3 = Colab + estas 2 GPU · 2 = solo Kaggle
PRIMER_TRABAJADOR = 2         # 2 si Colab es el 1 · 1 si trabajas solo en Kaggle
MONTAR_AQUI = False           # True si trabajas solo en Kaggle

import os, sys, subprocess, json
os.environ.update(VIDEOFREE_TRABAJO="/tmp/videofree/trabajo", VIDEOFREE_HERRAMIENTAS="/tmp/videofree", HF_HOME="/tmp/videofree/hf",
                  VIDEOFREE_REPO=REPO, VIDEOFREE_RAMA=RAMA)
os.makedirs("/tmp/videofree", exist_ok=True)
if os.path.exists("/tmp/videofree/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/tmp/videofree/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/tmp/videofree/VIDEO-FREE"], check=True)
sys.path.insert(0, "/tmp/videofree/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=False)
if comun._token:
    os.environ["GITHUB_TOKEN"] = comun._token  # para los dos trabajadores
GPUS = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print(len(GPUS), "GPU:", *GPUS, sep="\n  ")
OPCIONES = dict(proyecto=PROYECTO, formato=FORMATO, calidad=CALIDAD, ritmo=RITMO, semilla=SEMILLA, motor=MOTOR,
                modo_personaje=MODO_PERSONAJE, labios=LABIOS)

In [ ]:
# 2 · Instalar y descargar los modelos una sola vez para las dos GPU (8–15 min la primera vez)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft gguf ftfy sentencepiece imageio imageio-ffmpeg
import importlib, director, generar
importlib.reload(director); importlib.reload(generar)
from huggingface_hub import snapshot_download, hf_hub_download
SDXL = ["model_index.json", "scheduler/*", "tokenizer/*", "tokenizer_2/*", "*/config.json", "*.fp16.safetensors"]
snapshot_download(generar.INPAINT, allow_patterns=SDXL)
snapshot_download(generar.REALVIS_RAPIDO, allow_patterns=SDXL)
snapshot_download(generar.M5, allow_patterns=["model_index.json", "scheduler/*", "tokenizer/*", "text_encoder/*", "vae/*"])
gguf = generar.GGUF_FAST if MOTOR == "rapido" else generar.GGUF
repo_gguf, archivo_gguf = gguf.split("huggingface.co/")[1].split("/blob/main/")
hf_hub_download(repo_gguf, archivo_gguf)
if LABIOS:
    generar.Director.instalar_latentsync(vram=15)
print("Todo listo para las dos GPU")

In [ ]:
# 3 · Las dos GPU trabajan a la vez (cada una en sus tomas)
import time
procesos = []
for i in range(min(2, len(GPUS))):
    k = PRIMER_TRABAJADOR + i
    registro = open(f"/tmp/videofree/trabajador_{k}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i), "VIDEOFREE_TRABAJO": f"/tmp/videofree/trabajo_{k}"}
    procesos.append((k, subprocess.Popen([sys.executable, "-u", "/tmp/videofree/VIDEO-FREE/colab/generar.py", "--config", json.dumps(OPCIONES),
                                          "--trabajador", str(k), "--trabajadores", str(TRABAJADORES), "--repo", REPO, "--rama", RAMA],
                                         cwd="/tmp/videofree/VIDEO-FREE/colab", env=env, stdout=registro, stderr=subprocess.STDOUT)))
print("Trabajadores en marcha:", [k for k, _ in procesos])
while any(p.poll() is None for _, p in procesos):
    time.sleep(60)
    for k, p in procesos:
        ultimas = open(f"/tmp/videofree/trabajador_{k}.log").read().strip().splitlines()[-2:]
        print(f"[{time.strftime('%H:%M')}] trabajador {k} ({'trabajando' if p.poll() is None else 'terminó'}):", *ultimas, sep="\n   ")
for k, p in procesos:
    print(f"Trabajador {k}: {'bien' if p.returncode == 0 else 'falló, mira /tmp/videofree/trabajador_%d.log' % k}")

In [ ]:
# 4 · Montaje (solo si trabajas solo en Kaggle; con Colab, monta Colab)
if MONTAR_AQUI:
    d = generar.Director(generar.Config(**OPCIONES, trabajador=PRIMER_TRABAJADOR, trabajadores=TRABAJADORES), 15)
    video = d.montar(esperar_min=30)
    d.guardar(True)
    from IPython.display import Video, display
    display(Video(str(video), embed=True, width=720))
else:
    print("Listo. En Colab, la celda 7 del cuaderno 5 monta el video con las tomas de las 3 GPU.")